# 02 — Calendriers, registre et algèbre

477 calendriers sont embarqués dans la wheel. Ce notebook montre comment les trouver, ce
qu'ils savent sur eux-mêmes, et surtout comment les **combiner** — la partie où le
vocabulaire piège tout le monde.

In [1]:
import numpy as np
import pandas as pd

import better_calendar as bcal
from better_calendar import Calendar

print(f"{len(bcal.list())} calendriers résolvables")

479 calendriers résolvables


## 1. Trouver un calendrier

Les identifiants sont **namespacés**. Un nom nu de quatre lettres est un MIC ISO-10383 ;
tout le reste porte un préfixe, donc `XNYS` ne peut pas être confondu avec un pays.

In [2]:
from collections import Counter

manifeste = bcal.calendars.snapshot.load_manifest()
familles = Counter(
    nom.split(":")[0] if ":" in nom else "MIC (bourse)" for nom in bcal.list()
)
pd.DataFrame(sorted(familles.items(), key=lambda kv: -kv[1]), columns=["préfixe", "nombre"])

,préfixe,nombre
0,country,251
1,wk,76
2,ql,75
3,MIC (bourse),57
4,fin,12
5,rate,4
6,exchange,3
7,crypto,1


In [3]:
# Un échantillon de chaque famille.
for prefixe, exemples in [
    ("MIC", ["XNYS", "XLON", "XPAR", "XTKS", "XTAE"]),
    ("country:", ["country:FR", "country:US", "country:US-NY", "country:JP"]),
    ("fin:", ["fin:TARGET2", "fin:NYB", "fin:LNB"]),
    ("rate:", ["rate:SOFR", "rate:ESTR", "rate:SONIA"]),
    ("autres", ["crypto:24x7", "weekday", "ql:UnitedStates.NYSE", "wk:FR"]),
]:
    print(f"{prefixe:10s} {', '.join(exemples)}")

MIC        XNYS, XLON, XPAR, XTKS, XTAE
country:   country:FR, country:US, country:US-NY, country:JP
fin:       fin:TARGET2, fin:NYB, fin:LNB
rate:      rate:SOFR, rate:ESTR, rate:SONIA
autres     crypto:24x7, weekday, ql:UnitedStates.NYSE, wk:FR


Les alias couvrent les noms qu'on utilise vraiment à l'oral. Ils vivent dans une table
déclarative unique, jamais en dur dans le code.

In [4]:
lignes = []
for alias in ("NYSE", "NASDAQ", "TARGET", "EUR", "GBP", "USD", "SONIA", "ESTR", "FR", "CRYPTO"):
    calendrier = bcal.get(alias)
    lignes.append({"alias": alias, "résout vers": calendrier.name, "fériés": len(calendrier.holidays)})
pd.DataFrame(lignes).set_index("alias")

,résout vers,fériés
alias,,
NYSE,XNYS,1227
NASDAQ,XNYS,1227
TARGET,fin:TARGET2,536
EUR,fin:TARGET2,536
GBP,fin:LNB,1055
USD,fin:NYB,1376
SONIA,fin:LNB,1055
ESTR,fin:TARGET2,536
FR,country:FR,1120


In [5]:
# Un alias et sa cible sont le *même objet* — la mémoïsation traverse la résolution.
bcal.get("NYSE") is bcal.get("XNYS")

True

Un nom inconnu ne renvoie pas `None` : il lève, avec des suggestions.

In [6]:
try:
    bcal.get("XNYZ")
except bcal.UnknownCalendarError as exc:
    print(exc)

Unknown calendar 'XNYZ'. Did you mean: 'XNZE', 'XNYS'? Use better_calendar.list() to see everything available.


## 2. Provenance

Chaque calendrier sait d'où il vient : quelle source, quelle **version** de cette source,
sur quel horizon, et une empreinte du contenu. C'est ce qui rend une date de règlement
auditable six mois plus tard.

In [7]:
bcal.describe("rate:SOFR")

{'name': 'rate:SOFR',
 'weekmask': 'Mon Tue Wed Thu Fri',
 'bounds': ['1970-01-01', '2100-12-31'],
 'tz': None,
 'session_start': '00:00:00',
 'holidays': 1473,
 'business_days': 32704,
 'provider': 'quantlib',
 'provider_version': '1.43',
 'hash': 'e1ce5112f1eb5ae7302b35ed98698941bc0f40a9',
 'requested': 'rate:SOFR',
 'canonical': 'rate:SOFR'}

Le point important : `provider_version` est figée. Mettre à jour `QuantLib` sur la machine
ne change **rien** à la réponse — les données viennent d'un fichier commité, pas d'un appel
à la bibliothèque amont.

In [8]:
import sys

# Aucun fournisseur n'est importé, même après avoir interrogé 400 calendriers.
for nom in list(bcal.list())[:400]:
    bcal.get(nom)
[m for m in ("exchange_calendars", "holidays", "QuantLib", "workalendar") if m in sys.modules]

[]

## 3. Ce qu'un calendrier contient

In [9]:
nyse = bcal.get("XNYS")
print("nom          :", nyse.name)
print("weekmask     :", nyse.weekmask)
print("fuseau       :", nyse.tz)
print("bornes       :", nyse.bounds)
print("fériés       :", len(nyse.holidays))
print("jours ouvrés :", len(nyse.good_days()))
print("\n5 premiers fériés 2026 :")
print(list(nyse.holidays_between("2026-01-01", "2027-01-01")[:5].strftime("%Y-%m-%d (%a)")))

nom          : XNYS
weekmask     : Mon Tue Wed Thu Fri
fuseau       : America/New_York
bornes       : (datetime.date(1970, 1, 1), datetime.date(2100, 12, 31))
fériés       : 1227
jours ouvrés : 32950

5 premiers fériés 2026 :
['2026-01-01 (Thu)', '2026-01-19 (Mon)', '2026-02-16 (Mon)', '2026-04-03 (Fri)', '2026-05-25 (Mon)']


Tous les calendriers ne sont pas lundi–vendredi. Tel-Aviv cote du dimanche au jeudi, et
la weekmask est **déduite des séances réelles**, pas supposée :

In [10]:
tase = bcal.get("XTAE")
print("XTAE weekmask :", tase.weekmask)
print("dimanche 2 août 2026 ouvré ?", tase.is_bday("2026-08-02"))
print("vendredi 31 juillet 2026 ouvré ?", tase.is_bday("2026-07-31"))

XTAE weekmask : Mon Tue Wed Thu Sun
dimanche 2 août 2026 ouvré ? True
vendredi 31 juillet 2026 ouvré ? False


## 4. L'algèbre, et le piège de vocabulaire

**C'est la zone la plus casse-gueule de la bibliothèque.** « L'union de deux calendriers »
veut dire l'inverse selon qu'on pense en jours **ouvrés** ou en jours **fériés**.

La bibliothèque tranche : tout est nommé en jours ouvrés.

| Expression | Signification |
|---|---|
| `a & b` | ouvré dans **les deux** — donc l'**union des fériés** ← le cas règlement |
| `a \| b` | ouvré dans **au moins un** |
| `a - b` | ouvré dans `a`, pas dans `b` |
| `a ^ b` | ouvré dans **exactement un** |

Prenons deux dates où New York et la zone euro divergent : le 3 juillet 2026
(Independence Day observé, NY fermé) et le 6 avril 2026 (lundi de Pâques, TARGET2 fermé).

In [11]:
ny = bcal.get("XNYS")
eur = bcal.get("fin:TARGET2")

dates = ["2026-07-03", "2026-04-06", "2026-07-02"]
tableau = {
    "XNYS": {d: ny.is_bday(d) for d in dates},
    "fin:TARGET2": {d: eur.is_bday(d) for d in dates},
    "a & b  (les deux)": {d: (ny & eur).is_bday(d) for d in dates},
    "a | b  (au moins un)": {d: (ny | eur).is_bday(d) for d in dates},
    "a - b  (NY seul)": {d: (ny - eur).is_bday(d) for d in dates},
    "a ^ b  (exactement un)": {d: (ny ^ eur).is_bday(d) for d in dates},
}
pd.DataFrame(tableau).T

,2026-07-03,2026-04-06,2026-07-02
XNYS,False,True,True
fin:TARGET2,True,False,True
a & b (les deux),False,False,True
a | b (au moins un),True,True,True
a - b (NY seul),False,True,False
a ^ b (exactement un),True,True,False


Le cas concret : un flux de trésorerie entre New York et la zone euro ne peut bouger
qu'un jour où **les deux** places sont ouvertes. C'est `&`, et c'est bien l'union des
jours fériés.

Quelqu'un qui décrit ça comme « l'union des calendriers » pense en fériés et tendrait la
main vers `|`. D'où les alias verbeux, qui ne laissent aucun doute en relecture :

In [12]:
reglement = Calendar.all_open([ny, eur])
print("nom dérivé :", reglement.name)
print("2026-07-02 + 1 jour de règlement ->", reglement.offset("2026-07-02", 1))
print("identique à & ?", reglement == (ny & eur))

nom dérivé : (XNYS & fin:TARGET2)
2026-07-02 + 1 jour de règlement -> 2026-07-06
identique à & ? True


Un composite est un `Calendar` ordinaire : gelé, hachable, réutilisable dans les offsets
et les échéanciers. Ses bornes sont l'intersection de celles des opérandes.

In [13]:
composite = ny & eur & bcal.get("XLON")
print("nom     :", composite.name)
print("bornes  :", composite.bounds)
print("ouvrés  :", len(composite.good_days()))
print("hachable:", hash(composite) == hash(ny & eur & bcal.get("XLON")))

nom     : ((XNYS & fin:TARGET2) & XLON)
bornes  : (datetime.date(1970, 1, 1), datetime.date(2100, 12, 31))
ouvrés  : 32313
hachable: True


### Weekends hétérogènes

L'implémentation est de l'algèbre d'ensembles sur les jours ouvrés, jamais une fusion de
chaînes weekmask. C'est ce qui fait marcher gratuitement le croisement d'un calendrier
dimanche–jeudi avec un lundi–vendredi :

In [14]:
gulf = Calendar("gulf", weekmask="Sun Mon Tue Wed Thu")
lun_ven = bcal.get("weekday")

print("intersection :", (lun_ven & gulf).weekmask)   # le recouvrement
print("union        :", (lun_ven | gulf).weekmask)   # tout sauf le samedi
print()
for jour in ("2026-07-31", "2026-08-01", "2026-08-02"):
    quoi = pd.Timestamp(jour).strftime("%a")
    print(f"{jour} ({quoi})  &={(lun_ven & gulf).is_bday(jour)!s:5s}  |={(lun_ven | gulf).is_bday(jour)}")

intersection : Mon Tue Wed Thu
union        : Mon Tue Wed Thu Fri Sun

2026-07-31 (Fri)  &=False  |=True


2026-08-01 (Sat)  &=False  |=False
2026-08-02 (Sun)  &=False  |=True


### Fuseau d'un composite

Un composite qui enjambe deux fuseaux n'a **pas** de sémantique d'instant, donc il perd
son fuseau — et le dire tôt vaut mieux qu'une réponse fausse plus tard.

In [15]:
print("XNYS tz                :", ny.tz)
print("XPAR tz                :", bcal.get("XPAR").tz)
print("XNYS & XPAR            :", (ny & bcal.get("XPAR")).tz, "  <- désaccord : le fuseau tombe")

# Deux calendriers qui déclarent le *même* fuseau le conservent.
desk_ny = Calendar("desk:ny", tz="America/New_York", holidays=["2026-11-27"])
print("XNYS & desk:ny         :", (ny & desk_ny).tz)

# La comparaison porte sur le nom IANA, pas sur le décalage horaire : Paris et Amsterdam
# ont le même décalage mais deux identifiants distincts, donc le composite les départage.
print("XPAR & XAMS            :", (bcal.get("XPAR") & bcal.get("XAMS")).tz,
      f'  ({bcal.get("XPAR").tz} vs {bcal.get("XAMS").tz})')

XNYS tz                : America/New_York
XPAR tz                : Europe/Paris


XNYS & XPAR            : None   <- désaccord : le fuseau tombe
XNYS & desk:ny         : America/New_York
XPAR & XAMS            : None   (Europe/Paris vs Europe/Amsterdam)


## 5. Dériver un calendrier sans forker

`with_holidays` / `without_holidays` renvoient un **nouveau** calendrier. L'original est
gelé et ne bouge jamais.

In [16]:
desk = ny.with_holidays(["2026-11-27"], name="desk:us")   # lendemain de Thanksgiving
print("desk        :", desk.is_bday("2026-11-27"))
print("XNYS intact :", ny.is_bday("2026-11-27"))

desk        : False
XNYS intact : True


In [17]:
# Et on peut l'enregistrer sous un nom, pour que tous les call sites le trouvent.
bcal.register("desk:us", desk)
print(bcal.get("desk:us").is_bday("2026-11-27"))
print("listé :", "desk:us" in bcal.list())
bcal.unregister("desk:us")

False
listé : True


## Récapitulatif

| Appel | Rôle |
|---|---|
| `bcal.get(nom)` | résoudre un identifiant ou un alias, mémoïsé |
| `bcal.list(provider=)` | tout ce qui est résolvable |
| `bcal.describe(nom)` | provenance : source, version, bornes, empreinte |
| `bcal.register` / `unregister` | installer un calendrier maison |
| `a & b`, `a \| b`, `a - b`, `a ^ b` | algèbre, nommée en jours **ouvrés** |
| `Calendar.all_open` / `any_open` | les mêmes, en version non ambiguë |
| `with_holidays` / `without_holidays` | dériver sans muter |

**Suite :** [03 — Offsets, tenors et spot](03-offsets-tenors-spot.ipynb)